# Savepoints and Nested Transactions

## Overview
**Savepoints** allow partial rollback within a transaction — rolling back only a section of work without undoing the entire transaction. They are the SQL mechanism for nested-transaction behaviour.

**The problem savepoints solve:**
A batch payment job processes 100 transfers. Transfer #47 fails (insufficient funds). Without savepoints, the entire batch must be rolled back and retried. With savepoints, only transfer #47 is rolled back; the other 99 complete successfully.

**Savepoint syntax:**
```sql
BEGIN;
  -- ... some work ...
  SAVEPOINT my_checkpoint;       -- mark current state
  -- ... risky work ...
  ROLLBACK TO my_checkpoint;     -- undo risky work; outer transaction still open
  RELEASE my_checkpoint;         -- remove the savepoint marker
  -- ... more work ...
COMMIT;                          -- commit everything except what was rolled back
```

**Key distinction:**
- `ROLLBACK TO savepoint` — undo work back to the savepoint; transaction stays open
- `RELEASE savepoint` — remove the marker; does NOT commit or undo anything
- `ROLLBACK` (without TO) — undo the entire transaction including all savepoints
- `COMMIT` — commit everything; all savepoints are automatically released

**PostgreSQL PL/pgSQL:** The `EXCEPTION` block in a function implicitly uses a savepoint, allowing the function to catch errors and continue the outer transaction.

---

In [ ]:
import sqlite3

def get_conn():
    c = sqlite3.connect(":memory:")
    c.execute("PRAGMA journal_mode=WAL")
    c.row_factory = sqlite3.Row
    return c

conn = get_conn()
conn.executescript("""
CREATE TABLE accounts (
    account_id TEXT PRIMARY KEY,
    owner_name TEXT NOT NULL,
    balance    REAL NOT NULL CHECK(balance >= 0)
);
CREATE TABLE ledger (
    entry_id   INTEGER PRIMARY KEY AUTOINCREMENT,
    account_id TEXT NOT NULL,
    amount     REAL NOT NULL,
    entry_type TEXT NOT NULL,
    description TEXT,
    status     TEXT NOT NULL DEFAULT 'posted'
);
INSERT INTO accounts VALUES
    ('ACC001','Aroha Ngata', 5000.0),
    ('ACC002','Liam Chen',   2500.0),
    ('ACC003','Fatima Rashid',8000.0);
""")
conn.commit()

def show(c, msg):
    print(f"  {msg}")
    rows = c.execute("SELECT account_id, balance FROM accounts ORDER BY account_id").fetchall()
    for r in rows:
        print(f"    {r['account_id']}  ${r['balance']:,.2f}")
    count = c.execute("SELECT COUNT(*) FROM ledger").fetchone()[0]
    print(f"    ledger entries: {count}")

show(conn, "Initial state:")


---
## SAVEPOINT syntax and partial rollback

In [ ]:

print("=== SAVEPOINT: partial rollback within a transaction ===")
savepoint_demo = """
BEGIN;
-- Debit ACC001
UPDATE accounts SET balance = balance - 1000 WHERE account_id = 'ACC001';
INSERT INTO ledger(account_id, amount, entry_type, description)
    VALUES ('ACC001', 1000, 'debit', 'batch transfer out');

SAVEPOINT after_debit;                  -- checkpoint after the debit

-- Credit ACC002
UPDATE accounts SET balance = balance + 600  WHERE account_id = 'ACC002';
INSERT INTO ledger(account_id, amount, entry_type, description)
    VALUES ('ACC002', 600, 'credit', 'batch transfer in');

SAVEPOINT after_first_credit;          -- checkpoint after first credit

-- Credit ACC003 — this will fail (say, wrong account)
UPDATE accounts SET balance = balance + 500  WHERE account_id = 'ACC003';
INSERT INTO ledger(account_id, amount, entry_type, description)
    VALUES ('ACC003_WRONG', 500, 'credit', 'batch transfer in');  -- FK violates

-- On error: roll back only to the last savepoint, not the whole transaction
ROLLBACK TO after_first_credit;        -- undo ACC003 credit only
RELEASE after_first_credit;            -- release savepoint (optional cleanup)

-- Continue with corrected ACC003 credit
UPDATE accounts SET balance = balance + 400  WHERE account_id = 'ACC003';
INSERT INTO ledger(account_id, amount, entry_type, description)
    VALUES ('ACC003', 400, 'credit', 'corrected transfer in');

COMMIT;                                -- ACC001 debit + ACC002 credit + corrected ACC003 credit
-- If we had done ROLLBACK here instead, ALL changes since BEGIN would be undone.
"""
print(savepoint_demo)


---
## Savepoints in Python: batch processing with per-item rollback

In [ ]:

print("=== Savepoints in Python sqlite3 ===")

import sqlite3

def get_conn():
    c = sqlite3.connect(":memory:")
    c.execute("PRAGMA journal_mode=WAL")
    c.row_factory = sqlite3.Row
    return c

conn = get_conn()
conn.executescript("""
CREATE TABLE accounts (account_id TEXT PRIMARY KEY, owner_name TEXT NOT NULL,
    balance REAL NOT NULL CHECK(balance >= 0));
CREATE TABLE ledger (entry_id INTEGER PRIMARY KEY AUTOINCREMENT,
    account_id TEXT NOT NULL, amount REAL NOT NULL, entry_type TEXT NOT NULL,
    description TEXT);
INSERT INTO accounts VALUES
    ('ACC001','Aroha Ngata',5000.0),
    ('ACC002','Liam Chen',  2500.0),
    ('ACC003','Fatima Rashid',8000.0);
""")
conn.commit()

def process_batch_payments(conn, payments):
    """
    Process a batch of payments.
    Each payment is attempted independently within a savepoint.
    A failed payment is rolled back; others continue.
    """
    conn.execute("BEGIN")
    results = []
    for i, (from_id, to_id, amount, desc) in enumerate(payments):
        sp = f"sp_{i}"
        conn.execute(f"SAVEPOINT {sp}")
        try:
            conn.execute(
                "UPDATE accounts SET balance = balance - ? WHERE account_id = ?",
                (amount, from_id))
            conn.execute(
                "UPDATE accounts SET balance = balance + ? WHERE account_id = ?",
                (amount, to_id))
            conn.execute(
                "INSERT INTO ledger(account_id,amount,entry_type,description) VALUES(?,?,'debit',?)",
                (from_id, amount, desc))
            conn.execute(f"RELEASE SAVEPOINT {sp}")
            results.append((desc, "SUCCESS"))
        except Exception as e:
            conn.execute(f"ROLLBACK TO SAVEPOINT {sp}")
            conn.execute(f"RELEASE SAVEPOINT {sp}")
            results.append((desc, f"FAILED: {e}"))
    conn.execute("COMMIT")
    return results

payments = [
    ("ACC001", "ACC002", 300.0, "rent payment"),
    ("ACC001", "ACC003", 200.0, "utility payment"),
    ("ACC002", "ACC001", 9999.0, "overdraft attempt"),   # will fail CHECK constraint
    ("ACC003", "ACC002", 150.0, "loan repayment"),
]

results = process_batch_payments(conn, payments)
print("Batch payment results:")
for desc, status in results:
    print(f"  {desc:<22s}  {status}")

print()
print("Final balances:")
for row in conn.execute("SELECT account_id, owner_name, balance FROM accounts ORDER BY account_id"):
    print(f"  {row['account_id']}  {row['owner_name']:<14s}  ${row['balance']:,.2f}")
print("Ledger entries:", conn.execute("SELECT COUNT(*) FROM ledger").fetchone()[0])


---
## Nested transaction patterns and PostgreSQL notes

In [ ]:

print("=== Nested transaction patterns and PostgreSQL notes ===")
nested_notes = """
PostgreSQL does NOT support true nested transactions.
SAVEPOINT is the PostgreSQL mechanism for nested-transaction behaviour.

Common patterns:

1. Batch processing with per-item savepoints (shown above):
   BEGIN → [SAVEPOINT sp_0 → work → RELEASE sp_0] × N → COMMIT
   Use: process N items; failures roll back the individual item, not the batch.

2. Application-level nested transactions via savepoint stacks:
   Some ORMs (SQLAlchemy, Django) implement nested transactions using savepoints:
   outer_tx.begin()
       inner_tx.begin()   → creates SAVEPOINT
           ... work ...
       inner_tx.commit()  → RELEASE SAVEPOINT (if success) or ROLLBACK TO (if fail)
   outer_tx.commit()      → actual COMMIT

3. PostgreSQL EXCEPTION handling in PL/pgSQL:
   CREATE OR REPLACE FUNCTION safe_transfer(from_id TEXT, to_id TEXT, amount NUMERIC)
   RETURNS VOID AS $$
   BEGIN
       UPDATE accounts SET balance = balance - amount WHERE account_id = from_id;
       UPDATE accounts SET balance = balance + amount WHERE account_id = to_id;
   EXCEPTION
       WHEN check_violation OR foreign_key_violation THEN
           -- PostgreSQL auto-rolls back the subtransaction up to the function call
           RAISE NOTICE 'Transfer failed: %', SQLERRM;
   END;
   $$ LANGUAGE plpgsql;
   -- The EXCEPTION block internally uses a savepoint to allow partial rollback.

4. AUTONOMOUS TRANSACTIONS (PostgreSQL does not have them natively):
   -- Oracle has PRAGMA AUTONOMOUS_TRANSACTION.
   -- In PostgreSQL, use dblink or a separate connection to simulate
   -- writes that commit independently of the outer transaction.
   -- Use case: audit logging that must persist even if the outer tx rolls back.
"""
print(nested_notes)

print("SQLite savepoint syntax recap:")
sqlite_syntax = [
    ("SAVEPOINT name",           "Create a savepoint named 'name'"),
    ("RELEASE SAVEPOINT name",   "Remove savepoint (does NOT commit; just cleans up)"),
    ("ROLLBACK TO SAVEPOINT name","Roll back to savepoint (savepoint itself remains)"),
    ("ROLLBACK",                 "Roll back entire transaction including all savepoints"),
    ("COMMIT",                   "Commit entire transaction; all savepoints released"),
]
for syntax, meaning in sqlite_syntax:
    print(f"  {syntax:<38s}  {meaning}")


---
## Common Pitfalls

**1. Releasing a savepoint instead of rolling back to it on error**
`RELEASE SAVEPOINT sp` simply removes the savepoint marker without undoing any work. On error, the correct sequence is: `ROLLBACK TO SAVEPOINT sp` (undo the work) followed by `RELEASE SAVEPOINT sp` (clean up). Calling only `RELEASE` on an error path commits the failed partial work when the outer transaction eventually commits.

**2. Assuming savepoints create true nested transactions**
Rolling back to a savepoint does not commit the work done before the savepoint. It only undoes work done after the savepoint was created. The outer transaction is still open and must be explicitly committed or rolled back. Savepoints are partial-rollback checkpoints, not independent transactions.

**3. Savepoint names must be unique within a transaction**
In PostgreSQL, creating a savepoint with a name that already exists in the current transaction moves the savepoint to the current position (losing the old one). In SQLite, it creates a stack (inner RELEASE unwinds to the previous savepoint of the same name). Always use unique names per transaction, or use a counter: `sp_0`, `sp_1`, `sp_2`.

**4. Audit log inserts inside a rolling-back transaction**
If audit log inserts happen inside the same transaction as the work being audited, a rollback also undoes the audit entries. For audit logs that must persist even on failure, use a separate database connection (autonomous transaction pattern), a message queue, or a PostgreSQL `dblink` call that commits independently.

**5. Deeply nested savepoints causing excessive overhead**
Each savepoint requires the database to track the transaction state at that point. Thousands of savepoints inside one long transaction (e.g. one per row in a million-row batch) accumulate significant overhead. For very large batches, commit in sub-batches of 1,000-10,000 rows rather than using savepoints within a single mega-transaction.

**6. Not testing the savepoint rollback path in integration tests**
The savepoint rollback path -- where one item in a batch fails and the rest succeed -- is exactly the code path that is hardest to test and most likely to have bugs. Always write integration tests that deliberately trigger a savepoint rollback and verify that: (1) the failed item is not committed, (2) the successful items are committed, and (3) the database is in a consistent state after the partial failure.


---
*sql_methods_library - Samantha McGarrigle*